In [1]:
import pandas as pd
import altair as alt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

/Users/mishaankud/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/var/folders/s7/h860_dc93613dqrkg9qpvl240000gn/T/ipykernel_22894/492662372.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
df = pd.read_csv('../data/data.csv')

In [3]:
# Make a dataframe for each decade
twenties_df = df[(df['year'] >= 1920) & (df['year'] <= 1929)]
thirties_df = df[(df['year'] >= 1930) & (df['year'] <= 1939)]
forties_df = df[(df['year'] >= 1940) & (df['year'] <= 1949)]
fifties_df = df[(df['year'] >= 1950) & (df['year'] <= 1959)]
sixties_df = df[(df['year'] >= 1960) & (df['year'] <= 1969)]
seventies_df = df[(df['year'] >= 1970) & (df['year'] <= 1979)]
eighties_df = df[(df['year'] >= 1980) & (df['year'] <= 1989)]
nineties_df = df[(df['year'] >= 1990) & (df['year'] <= 1999)]
aughts_df = df[(df['year'] >= 2000) & (df['year'] <= 2009)]
tens_df = df[(df['year'] >= 2010) & (df['year'] <= 2019)]
now_df = df[df['year'] >= 2020]

In [4]:
# Make each decade dataframe only the top 10% popularity scores
twenties_df = twenties_df[twenties_df['popularity'] >= twenties_df['popularity'].quantile(0.9)]
thirties_df = thirties_df[thirties_df['popularity'] >= thirties_df['popularity'].quantile(0.9)]
forties_df = forties_df[forties_df['popularity'] >= forties_df['popularity'].quantile(0.9)]
fifties_df = fifties_df[fifties_df['popularity'] >= fifties_df['popularity'].quantile(0.9)]
sixties_df = sixties_df[sixties_df['popularity'] >= sixties_df['popularity'].quantile(0.9)]
seventies_df = seventies_df[seventies_df['popularity'] >= seventies_df['popularity'].quantile(0.9)]
eighties_df = eighties_df[eighties_df['popularity'] >= eighties_df['popularity'].quantile(0.9)]
nineties_df = nineties_df[nineties_df['popularity'] >= nineties_df['popularity'].quantile(0.9)]
aughts_df = aughts_df[aughts_df['popularity'] >= aughts_df['popularity'].quantile(0.9)]
tens_df = tens_df[tens_df['popularity'] >= tens_df['popularity'].quantile(0.9)]
now_df = now_df[now_df['popularity'] >= now_df['popularity'].quantile(0.9)]

In [5]:
# Make dataframe of the mean attributes for each decade

decade_dfs = [twenties_df, thirties_df, forties_df, fifties_df, sixties_df, seventies_df, eighties_df, nineties_df, aughts_df, tens_df, now_df]
decades = [1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
attributes = ['speechiness', 'acousticness', 'liveness', 'danceability', 'valence', 'instrumentalness']

decadesmeans = {}
# Iterate through decades using i to reference decade_dfs and decades lists
for i in range(len(decades)):
    # Create a dictionary of the means for the decade
    row = {}
    for attribute in attributes:
            row[attribute] = decade_dfs[i][attribute].mean()
    # Append decade means dict to a dict of dicts of each decade, using decade as key
    decadesmeans[decades[i]] = row

# Convert dict of dicts to dataframe and convert attribute from index to a column
decadesmeans_df = pd.DataFrame(decadesmeans).reset_index(names='attribute')
# Melt dataframe for plotting
decadesmeans_df = decadesmeans_df.melt(id_vars=['attribute'], var_name='decade')
decadesmeans_df

,attribute,decade,value
0,speechiness,1920,0.087582
1,acousticness,1920,0.942866
2,liveness,1920,0.199008
3,danceability,1920,0.612006
4,valence,1920,0.607642
...,...,...,...
61,acousticness,2020,0.208012
62,liveness,2020,0.169086
63,danceability,2020,0.720524
64,valence,2020,0.522978


In [6]:
# Create slider selection for decades
slider = alt.binding_range(min = 1920, max = 2020, step = 10, name = 'Decade: ')
selection = alt.selection_point(fields = ['decade'], bind = slider, value = 2020)

# Define a custom color scale to make each bar a different color to match the other figures
color_scale = alt.Scale(range=['#4C78A8', '#F58517', '#72B7B2', '#53A24A', '#EECA3B', '#B379A2'])

# Create the bar plot as a overlapping bar plot of each decades attribute, only showing selected decade
chart = alt.Chart(decadesmeans_df).mark_bar().encode(
    alt.Y(
        'attribute:O',
        title='Musical Attributes',
        sort=['speechiness', 'acousticness', 'liveness', 'danceability', 'valence', 'instrumentalness']),
    alt.X(
        'value:Q',
        title='Average Value',
        stack=None,
        scale=alt.Scale(domain=[0, 1])),
        color = alt.Color('attribute:O', scale=color_scale, legend=None),
        opacity = alt.condition(selection, alt.value(1), alt.value(0))
).properties(
    title="Average Musical Attributes of Decades' Most Popular Songs",
    height=300, width='container'
).add_params(selection)
chart.autosize = {"type": "fit", "contains": "padding"}

In [7]:
# Save figure as html file
chart.save('../docs/ChartFiles/bar.html')